In [5]:
import os
import time
import re

DIR_PATH = './monitor_directory'

pre_file = set(os.listdir(DIR_PATH))
print(f"초기 파일: {pre_file}")

danger_ext = ('.py', '.js', '.class')

comment_pattern = r'(#.*$|//.*$|/\*[\s\S]*?\*/)' # "#" , //, /**/ 주석 패턴 감지
email_pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}' # 이메일 패턴 감지
sql_pattern = r'(?i)\b(SELECT|UPDATE|INSERT|DELETE|DROP|UNION|ALTER|TRUNCATE|EXEC|CREATE|MERGE)\b' #SQL 문 감지

# ==========================================
# [공통 코드] 파일 본문 보안 분석 로직
# ==========================================
def analyze_file_security(file_path):
    """
    지정한 파일의 본문을 읽어 주석, 이메일, SQL 구문 등 민감 정보를 탐지합니다.
    indent 인자를 통해 출력 포맷(공백 수)을 유연하게 조절할 수 있습니다.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        if re.search(comment_pattern, content, re.MULTILINE):
            print("주석 탐지")
        
        if re.search(email_pattern, content):
            print("이메일 탐지")
        
        if re.search(sql_pattern, content):
            print("SQL문 탐지")
    except Exception as e:
        print(f"파일 읽기 오류: {e}")

# ==========================================
# 1. 초기 파일 점검 로직
# ==========================================
for file in pre_file:
    # 조건 1: 주의 확장자 검사
    if file.endswith(danger_ext):
        print(f"주의 파일 탐지 : {file}")

        # 조건 2: 함수를 호출하여 민감 정보 검사
        file_path = os.path.join(DIR_PATH, file)
        analyze_file_security(file_path)

print("=== 초기 점검 완료 ===")

# ==========================================
# 2. 실시간 모니터링 루프
# ==========================================
try:
    while True:
        current_file = set(os.listdir(DIR_PATH))

        # 차집합 계산
        result_file = current_file - pre_file
        remove_file = pre_file - current_file

        # 변경사항이 있을 때만 화면에 출력하여 가독성 확보
        if result_file or remove_file:
            print(f"\n[+] 디렉터리 상태 변경 감지 ({time.strftime('%H:%M:%S')})")

            # 신규 파일 탐지
            for file in result_file:
                print(f"  * 새로운 파일 생성됨: {file}")

                # 주의 확장자 검사
                if file.endswith(danger_ext):
                    print(f"    -> [경고] 주의 확장자군 분류 위험 파일입니다.")

                # 함수를 호출하여 본문 보안 분석
                file_path = os.path.join(DIR_PATH, file)
                analyze_file_security(file_path)
            
            # 삭제 파일 탐지
            for file in remove_file:
                print(f"  * 파일 삭제됨: {file}")

            print("-" * 40)
        
        # 상태 업데이트 및 대기
        pre_file = current_file
        time.sleep(5) 

except KeyboardInterrupt:
    print("\n모니터링 시스템이 사용자에 의해 안전하게 종료되었습니다.") 


초기 파일: {'backup_query.sql', 'LegacyApp.class', 'config.json', 'auth_service.js', 'README.md', 'utils.py', 'test_script.py', 'main.css'}
주의 파일 탐지 : LegacyApp.class
주석 탐지
이메일 탐지
SQL문 탐지
주의 파일 탐지 : auth_service.js
주석 탐지
이메일 탐지
SQL문 탐지
주의 파일 탐지 : utils.py
주석 탐지
주의 파일 탐지 : test_script.py
주석 탐지
이메일 탐지
=== 초기 점검 완료 ===

[+] 디렉터리 상태 변경 감지 (09:41:33)
  * 새로운 파일 생성됨: a.py
    -> [경고] 주의 확장자군 분류 위험 파일입니다.
----------------------------------------

[+] 디렉터리 상태 변경 감지 (09:41:43)
  * 파일 삭제됨: a.py
----------------------------------------

모니터링 시스템이 사용자에 의해 안전하게 종료되었습니다.
